# Programmatic Magnetic Resonance Fingerprinting Optimization

### Orthogonality 📐

The accuracy and reliability of parameter estimation in MRF fundamentally depend on how distinguishable different signal fingerprints are from one another, leading to the critical aspect of signal orthogonality. When signal evolutions corresponding to different tissue parameters are highly orthogonal (i.e., minimally correlated), the matching process can more reliably distinguish between tissues with similar properties. Lower signal orthogonality for signal fingerprints with different relaxometric origins can lead to increased parameter estimation errors and reduced robustness to noise.


### Cramér-Rao Lower Bound (CRLB) ⛓

The Cramér-Rao Lower Bound (CRLB) provides a theoretical framework for understanding the best possible precision achievable in parameter estimation. In the context of MRF, the CRLB quantifies the minimum variance (uncertainty) in estimating parameters from the signal vectors. The inverse of the Fisher Information Matrix yields the CRLB, with diagonal elements representing the variance lower bounds for each parameter. Importantly, off-diagonal elements reveal correlations between parameter estimates. When parameters are highly correlated, they cannot be estimated independently with high precision.

To optimize MRF acquisition parameters, we require a forward model that can accurately simulate the complex signal evolution during the sequence. The Extended Phase Graph (EPG, see e.g. [Weigel 2015](https://doi.org/10.1002/jmri.24619)) formalism provides an elegant and computationally efficient solution for this purpose.

First we set up the necessary code and jupyter environment.

In [10]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as wgt
import tqdm.auto as tqdm
import rich

from IPython.display import display
from pathlib import Path
from numpy.typing import NDArray

In [11]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


This should be readily importable when the repository is cloned from GitHub and the notebook is run in its repository defined folder 🚀

In [ ]:
src_directory = Path('../src')
init_directory = src_directory / 'initialization'
sys.path.append(str(src_directory))
print = rich.print # nicer outputs

In [ ]:
import seqmetrics # noqa: F401
from plotting.splinetools import SplineSettingsDashboard
from plotting.curveditor import MonotonicCurveEditor
from plotting.simulator import SimulationController, SimulationControllerDashboard, SequenceParameters
from plotting.scattercanvas import InteractiveRelaxometricParameterCanvas
from plotting.signaldisplay import SignalDisplayDashboard, SignalDisplay, wire_callbacks
from plotting.historyplot import HistoryPlot
from slsqp import optimize_sequence

Here we load the ncessary flip angle train data and the repetition time pattern from preset numpy arrays.
Both patterns were deduced from the brain-specific work by [Cao et al. 2022](https://doi.org/10.1002/mrm.29194).
They act as an starting point for the interactive modification tooling and subsequently the optimization.
In principle, any other patterns can be loaded and tested here.
Other examples include the:
- original sinusoidal pattern by [Yun et al. 2015](https://doi.org/10.1002/mrm.25559)
- optimized patterns by [Zhao et al. 2019](10.1109/TMI.2018.2873704)

In [8]:
FA_DATA_PATH = init_directory / 'fa_cao.npy'
TR_DATA_PATH = init_directory / 'tr_cao.npy'
fa = np.load(FA_DATA_PATH)
tr = np.load(TR_DATA_PATH)
fa_initial_y = fa
fa_initial_x = np.arange(len(fa))
tr_initial_y = tr
tr_initial_x = np.arange(len(tr))

In [9]:
dashboard_fa = SplineSettingsDashboard.from_FA_defaults()
dashboard_tr = SplineSettingsDashboard.from_TR_defaults()

seqparams = SequenceParameters(
    ph=np.full_like(fa, fill_value=0.0),
    shots=len(fa),
    prep=[1],
    t2te=[0.0],
    ti=[10.0],
    te=1.0
)

with plt.ioff():
    fig, axes = plt.subplots(ncols=3, figsize=(13.3, 3.9))
    ce_fa = MonotonicCurveEditor(fa_initial_x, fa_initial_y, dashboard_fa, fig=fig, ax=axes[0])
    ce_fa.ax.set_ylabel('Flip Angle (degrees)')
    ce_fa.ax.set_title('Interactive Flip Angle Editor')

    ce_tr = MonotonicCurveEditor(tr_initial_x, tr_initial_y, dashboard_tr, fig=fig, ax=axes[1], initial_yaxis_range=(0, 100))
    ce_tr.ax.set_ylabel('Repetition Time (ms)')
    ce_tr.ax.set_title('Interactive Repetition Time Editor')

    cv = InteractiveRelaxometricParameterCanvas.prepopulated(species={'csf', 'wm', 'gm', 'muscle'}, ax=axes[2], fig=fig)

    ctr_dashboard = SimulationControllerDashboard()

    controller = SimulationController(
        dashboard=ctr_dashboard,
        fa_provider=ce_fa,
        tr_provider=ce_tr,
        relax_provider=cv,
        parameters=seqparams
    )

    tabs = wgt.Tab(
        children=[ctr_dashboard.ui, dashboard_fa.ui, dashboard_tr.ui],
        titles=['Simulation Dashboard', 'FA Settings', 'TR Settings'],
        style={'description_width': 'initial'}
    )

    _ = fig.tight_layout()
    
controller.run()

NameError: name 'SplineSettingsDashboard' is not defined

In [ ]:
dashboard = SignalDisplayDashboard.create()
sigdisp = SignalDisplay()
sigdisp.fig.update_layout(height=400, width=1400, showlegend=False)
sigdisp.fig.update_layout(xaxis=dict(title='Time (ms)'), yaxis=dict(title='Signal Amplitude (a.u.)'))

sigdisp.add_traces(controller.fetch_simulation_package(), dashboard.query_state())
wire_callbacks(sigdisp, dashboard, controller)

In [ ]:
display(
    wgt.VBox([
        wgt.VBox([tabs, fig.canvas]),
        dashboard.ui,
        sigdisp.fig
    ])
)

With the setup about the relaxometric species from the scatter canvas and the sequence specific parameters {flip angle train, repetition times} from the interactive spline curve obtained above, we can optimize the sequence with respect to different cost functions.

The parameter data defined by the points inside the relaxometric species canvas is reused for the optimization.
Note that the optimization does depend on the currently set species, since the output signal fitness is quantified via the orthogonality cost function.
In turn, this means that we can optimize for specific anatomic regions with prior knowledge about expected tissue
environments and subsequent relaxometric parameters.

In [ ]:
T1: NDArray = np.asarray([p.x for p in cv.get_points()], dtype=np.float32)
T2: NDArray = np.asarray([p.y for p in cv.get_points()], dtype=np.float32)
M0: float = 1.0

In [21]:
# Data from the sequence parameter spline curve editors
fa: NDArray = ce_fa.get_current_curve().y
tr: NDArray = ce_tr.get_current_curve().y
ph: float = seqparams.ph

In [22]:
# Optimization constraints for the flip angles
fa_min: float = 5.0
fa_max: float = 90.0
fa_maxdiff: float = 5.0


In [23]:
# Other sequence parameters (already defined above, but extracted here for clarity)
beats: int = seqparams.beats
shots: int = seqparams.shots
prep: list[int] = seqparams.prep
ti: list[float] = seqparams.ti
t2te: list[float] = seqparams.t2te
te: float = seqparams.te

In [19]:
fig.get_size_inches()

array([13.32,  3.9 ])

In [7]:
init_fa = ce_fa.get_current_curve().y

hp = HistoryPlot(max_TTL=40)
hp.add_immortal_trace(init_fa, width=1, color='green', dash='dot', opacity=0.7)
hp.fig.update_layout(xaxis=dict(title='TR index'), yaxis=dict(title='Flip Angle (degrees)'))

NameError: name 'ce_fa' is not defined

In [5]:
n_max_iter: int = 40

pbar = tqdm.tqdm(total=n_max_iter)

def callback(x: np.ndarray):
    # tgriesler optimizatiopn: single array for fa and tr
    # only propagate flip angles into visualization
    hp.add_trace(x[:x.size//2])
    pbar.update(1)

result = optimize_sequence(
    costfunction='orth_epg',
    t1=T1,
    t2=T2,
    m0=M0,
    beats=beats,
    shots=shots,
    fa=init_fa,
    tr=tr,
    ph=ph,
    prep=prep,
    ti=ti,
    t2te=t2te,
    te=te,
    fa_min=fa_min,
    fa_max=fa_max,
    fa_maxdiff=fa_maxdiff,
    n_iter_max=n_max_iter,
    callback=callback,
    iprint=0
)

  0%|          | 0/40 [00:00<?, ?it/s]

NameError: name 'optimize_sequence' is not defined

In [6]:

class OptimizationPackage:
    """
    Compound data structure to hold all relevant information about an
    optimization run.

    Attributes should respect JSON-serializability for easy saving/loading.
    """
    costfunction: str
    T1: list

In [ ]:
DATA = [t.y for t in hp.fig.data]


In [ ]:
import attrs


def to_list(arraylike: ArrayLike) -> list[float]:
    pass


@attrs.define
class OptimizationPackage:
    """
    Compound data structure to hold all relevant information about an
    optimization run.

    Attributes should respect JSON-serializability for easy saving/loading.
    """
    costfunction: str
    T1: list[float]
    T2: list[float]
    M0: float
    beats: int
    shots: int
    tr: list[float]
    fa_initial: list[float]
    fa_history: list[list[float]]
    ph: list[float]
    prep: list[int]
    ti: list[float]
    t2te: list[float]
    te: float
    fa_min: float
    fa_max: float
    fa_maxdiff: float
    n_iter_max: int

    @classmethod
    def from_preassembled(
        cls,
        costfunction: str,
        T1: NDArray,
        T2: NDArray,
        M0: float,
        sequence_params: SequenceParameters,
        fa_initial: NDArray,
        fa_history: list[NDArray],
        ) -> "OptimizationPackage":
        """
        Convenience constructor that deduces attributes from preassembled sequence
        parameters.
        """


  0%|          | 0/200 [00:00<?, ?it/s]

  NIT    FC           OBJFUN            GNORM
    1     2     2.373589E+03     1.133953E+00
    2     3     2.368923E+03     1.050547E+00
    3     4     2.366167E+03     7.486093E-01
    4     5     2.364338E+03     5.988088E-01
    5     6     2.362691E+03     5.236920E-01
    6     7     2.361469E+03     4.686596E-01
    7     8     2.360168E+03     4.120978E-01
    8     9     2.358962E+03     3.540827E-01
    9    10     2.358082E+03     2.897535E-01
   10    11     2.357598E+03     2.320233E-01
   11    12     2.357335E+03     1.953660E-01
   12    13     2.357025E+03     1.744273E-01
   13    14     2.356877E+03     1.522832E-01
   14    15     2.356603E+03     1.457189E-01
   15    16     2.356423E+03     1.453131E-01
   16    17     2.356053E+03     1.494561E-01
   17    18     2.355547E+03     1.455121E-01
   18    19     2.355401E+03     1.646963E-01
   19    20     2.355082E+03     1.914396E-01
   20    21     2.354492E+03     2.122671E-01
   21    22     2.354127E+03     1

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f0dbf6e0650>>
Traceback (most recent call last):
  File "/home/jannik/storage/esmrmb-notebook/.venv/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


   69    70     2.330409E+03     1.870680E-01
   70    71     2.330014E+03     1.864428E-01
   71    72     2.329686E+03     1.848286E-01
